# Sistema RAG Local — Perguntas & Respostas com Llama 3.2 + ChromaDB

**Cadeira:** Computação em Linguagem Natural (CLN) — UBI · MEI · 2025/2026
**Trabalho Final:** *Criação de Sistemas que Integram Linguagem Natural*
**Docente:** João Cordeiro

---

## 1. Objetivo

Construir um sistema de **RAG (Retrieval-Augmented Generation)** totalmente **local** que:

1. Recebe um **URL** (definido logo no início do notebook);
2. Faz **web scraping** desse URL e extrai o conteúdo textual;
3. Divide o texto em *chunks*, gera **embeddings** e indexa-os numa base de dados vetorial **ChromaDB** persistida em disco;
4. Responde a perguntas com o modelo **Llama 3.2** (via **Ollama**), usando *exclusivamente* o contexto recuperado da base — **reduzindo alucinações**.

## 3. Mapeamento às aulas teóricas

| Conceito aplicado | Aula |
|---|---|
| Embeddings densos/contextuais, *Information Retrieval*, motivação do RAG (alucinações, dados proprietários/dinâmicos) | **T09** |
| Pipeline RAG (*Loading → Chunking → Embedding → Vector Store*), *Sentence Embeddings*, LangChain, *web scraping* | **T10** |
| Llama 3.2 (Ollama) + ChromaDB local, cadeia LCEL, *loaders* e *text splitters* | **T11** |

## Arquitetura

```
        ┌───────────────────────  INGESTÃO (1x, requer Internet)  ──────────────────────┐
        │                                                                                │
  URL ──►  [Etapa 2]          [Etapa 3]           [Etapa 4]            [Etapa 5]          │
        │  Web Scraping  ──►   Chunking     ──►    Embeddings    ──►    ChromaDB          │
        │  requests+bs4        Recursive...        Sentence-Trf        (persist. disco)   │
        └────────────────────────────────────────────────────────────────────────────────┘
                                                                            │
        ┌──────────────────────  OPERAÇÃO (offline, sem APIs)  ─────────────┼─────────────┐
        │                                                                   ▼             │
  Pergunta ─►  [Etapa 6] Retriever  ──►  top-k chunks  ──►  [Etapa 8] Prompt + Llama 3.2  │
        │                                                        (Etapa 7, via Ollama)    │
        │                                                              │                  │
        └──────────────────────────────────────────────────────────►  ▼  Resposta ancorada
                                                                      (com fontes citadas)
```

## Pré-requisitos

Este sistema usa o **Ollama** para correr o Llama 3.2 localmente.

1. **Instalar o Ollama:** https://ollama.com/download

In [19]:
%pip install -q langchain langchain-core langchain-community langchain-chroma langchain-ollama langchain-huggingface langchain-text-splitters sentence-transformers chromadb beautifulsoup4 requests lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Etapa 0 — Configuração

In [20]:
# ===== FONTE DE CONHECIMENTO =====
URL = "https://pt.wikipedia.org/wiki/Processamento_de_linguagem_natural"

# ===== Modelos =====
OLLAMA_MODEL    = "llama3.2"                  # LLM gerador (via Ollama)
OLLAMA_BASE_URL = "http://localhost:11434"    # servidor local do Ollama
# Embeddings locais. Muito parecido com o MiniLM da T10.
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# ===== Base de dados vetorial local (ChromaDB) =====
CHROMA_DIR      = "chroma_db"     # nome da pasta
COLLECTION_NAME = "rag_web"       # nome da coleção

# ===== Parâmetros de chunking =====
CHUNK_SIZE      = 1000
CHUNK_OVERLAP   = 100

# ===== Recuperação =====
TOP_K           = 4               # nº de chunks recuperados por pergunta

## Etapa 1 — Verificação do ambiente

É preciso confirmar que o **Ollama** está acessível e que o **modelo** está
descarregado localmente.

In [21]:
import requests

def verificar_ollama(base_url=OLLAMA_BASE_URL, modelo=OLLAMA_MODEL):
    # Pergunta ao servidor local quais os modelos instalados.
    try:
        r = requests.get(f"{base_url}/api/tags", timeout=5)
        r.raise_for_status()
        instalados = [m.get("name", "") for m in r.json().get("models", [])]
        print("OK — Ollama acessível em", base_url)
        print("Modelos instalados:", instalados or "(nenhum)")
        if not any(modelo in nome for nome in instalados):
            print(f"AVISO: o modelo '{modelo}' não está instalado. Corre:  ollama pull {modelo}")
        else:
            print(f"OK — modelo '{modelo}' disponível.")
    except Exception as e:
        print("Ollama NÃO está acessível. Para corrigir:")
        print("   1) Instala o Ollama:        https://ollama.com/download")
        print(f"   2) Descarrega o modelo:     ollama pull {modelo}")
        print("   3) Arranca o servidor:      ollama serve")
        print("Detalhe técnico:", e)

verificar_ollama()

OK — Ollama acessível em http://localhost:11434
Modelos instalados: ['llama3.2:latest']
OK — modelo 'llama3.2' disponível.


## Etapa 2 — Web Scraping

Usamos `requests` + `BeautifulSoup`, removendo
elementos não informativos (scripts, navegação, rodapés) e normalizando o texto.

In [22]:
import re
import requests
from bs4 import BeautifulSoup

# Cabeçalho de browser para evitar bloqueios.
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/124.0 Safari/537.36"
}

def extrair_conteudo(url, timeout=15):
    # 1) Pedido HTTP
    resp = requests.get(url, headers=HEADERS, timeout=timeout)
    resp.raise_for_status()
    resp.encoding = resp.apparent_encoding   # deteta a codificação correta

    # 2) Parsing do HTML
    soup = BeautifulSoup(resp.text, "html.parser")

    # 3) Remove blocos sem valor informativo.
    for tag in soup(["script", "style", "nav", "footer", "header",
                     "aside", "noscript", "form"]):
        tag.decompose()

    titulo = soup.title.get_text(strip=True) if soup.title else url

    # 4) Prefere o corpo principal do artigo se existir
    principal = soup.find("main") or soup.find("article") or soup.body or soup
    texto = principal.get_text(separator="\n")

    # 5) Normaliza espaços em branco.
    linhas = [l.strip() for l in texto.splitlines()]
    linhas = [l for l in linhas if l]
    texto = "\n".join(linhas)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return titulo, texto

titulo, conteudo = extrair_conteudo(URL)
print("Título:", titulo)
print("Caracteres extraídos:", len(conteudo))
print("-" * 80)
print(conteudo[:1500], "...")

Título: Processamento de linguagem natural – Wikipédia, a enciclopédia livre
Caracteres extraídos: 34842
--------------------------------------------------------------------------------
Origem: Wikipédia, a enciclopédia livre.
As referências deste artigo
necessitam de formatação
.
Por favor, utilize
fontes apropriadas
contendo título, autor e data para que o verbete permaneça
verificável
.
(
maio de 2022
)
Este artigo carece de
reciclagem
de acordo com o
livro de estilo
.
Sinta-se livre para editá-lo(a) para que este(a) possa atingir um
nível de qualidade superior
.
(
dezembro de 2016
)
Esta página
cita fontes
, mas que
não cobrem
todo o conteúdo
.
Ajude a
inserir referências
(
Encontre fontes:
Google
(
N
•
L
•
A
•
I
•
WP refs
)
•
ABW
•
CAPES
).
(
dezembro de 2016
)
Parte da
série
sobre
Inteligência artificial (IA)
Objetivos
Inteligência artificial geral
Agente inteligente
Autoaperfeiçoamento recursivo
Planejamento automatizado
Visão computacional
Representação de conhecimento
Processa

## Etapa 3 — Chunking (segmentação do texto)

Os documentos têm de ser partidos em segmentos menores antes de indexar. Usamos o
`RecursiveCharacterTextSplitter` com `chunk_size=1000` e `chunk_overlap=100`.

In [23]:
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Documento inicial com metadados de origem (para depois citar a fonte).
doc_base = Document(page_content=conteudo, metadata={"source": URL, "title": titulo})

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
)
chunks = splitter.split_documents([doc_base])

# Identificador por chunk
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i

print("Número de chunks:", len(chunks))
print("-" * 80)
print("Pré-visualização do chunk 0:\n")
print(chunks[0].page_content[:500], "...")

Número de chunks: 43
--------------------------------------------------------------------------------
Pré-visualização do chunk 0:

Origem: Wikipédia, a enciclopédia livre.
As referências deste artigo
necessitam de formatação
.
Por favor, utilize
fontes apropriadas
contendo título, autor e data para que o verbete permaneça
verificável
.
(
maio de 2022
)
Este artigo carece de
reciclagem
de acordo com o
livro de estilo
.
Sinta-se livre para editá-lo(a) para que este(a) possa atingir um
nível de qualidade superior
.
(
dezembro de 2016
)
Esta página
cita fontes
, mas que
não cobrem
todo o conteúdo
.
Ajude a
inserir referências
( ...


## Etapa 4 — Embeddings locais

Convertemos cada chunk num vetor numérico que captura o seu **significado**. Usamos um modelo
**Sentence-Transformers** que corre **localmente**.

In [24]:
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Sanidade: dimensão do vetor produzido.
vec = embeddings.embed_query("Teste de embedding.")
print("Modelo de embeddings:", EMBEDDING_MODEL)
print("Dimensão do vetor:", len(vec))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Dimensão do vetor: 384


## Etapa 5 — Indexação no ChromaDB

Os vetores e o texto na **ChromaDB** são guardados e persistem em disco via SQLite

In [36]:
from langchain_chroma import Chroma

# IMPORTANTE: Chroma.from_documents ACRESCENTA à coleção; não a substitui.
# Por isso, primeiro removemos a coleção anterior (se existir) — torna a
# indexação idempotente: o resultado é sempre uma única cópia de cada chunk.
try:
    Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=CHROMA_DIR,
    ).delete_collection()
    print(f"Coleção '{COLLECTION_NAME}' anterior removida (evita duplicados).")
except Exception:
    pass  # ainda não existia — primeira indexação

# Cria a coleção do zero e indexa os chunks (gera embeddings + guarda em disco).
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
)

# Verificação: o total na base deve ser igual ao nº de chunks (sem duplicados).
total_na_base = vector_store._collection.count()
print(f"Indexados {len(chunks)} chunks na coleção '{COLLECTION_NAME}'.")
print(f"Total de documentos na base: {total_na_base}  (esperado: {len(chunks)})")
print(f"Base de dados persistida em: ./{CHROMA_DIR}")

Coleção 'rag_web' anterior removida (evita duplicados).
Indexados 43 chunks na coleção 'rag_web'.
Total de documentos na base: 43  (esperado: 43)
Base de dados persistida em: ./chroma_db


## Etapa 6 — Retriever (recuperação dos chunks relevantes)

O `as_retriever(search_kwargs={"k": TOP_K})` devolve os `k` chunks mais
semelhantes à pergunta, por similaridade no espaço de embeddings. Abaixo é mostrado
quais chunks são recuperados.

In [35]:
retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K})

# Demonstração: chunks recuperados para uma consulta de exemplo.
consulta_exemplo = "Qual é o tema principal do documento?"
recuperados = retriever.invoke(consulta_exemplo)

print("Consulta:", consulta_exemplo, "\n")
for d in recuperados:
    print(f"[chunk {d.metadata.get('chunk_id')}] {d.page_content[:200]} ...")
    print("-" * 80)

Consulta: Qual é o tema principal do documento? 

[chunk 0] Origem: Wikipédia, a enciclopédia livre.
As referências deste artigo
necessitam de formatação
.
Por favor, utilize
fontes apropriadas
contendo título, autor e data para que o verbete permaneça
verific ...
--------------------------------------------------------------------------------
[chunk 30] [
12
]
Em 1987, a primeira campanha de avaliação de textos escritos parece ser uma campanha dedicada à compreensão da mensagem (Pallet, 1998).
O projeto Parseval / GEIG comparou gramáticas de frase-es ...
--------------------------------------------------------------------------------
[chunk 17] Análise do Discurso
Esta rubrica inclui uma série de tarefas relacionadas. Uma tarefa é identificar a estrutura discursiva do texto conectado, isto é, a natureza das relações discursivas entre sentenç ...
--------------------------------------------------------------------------------
[chunk 23] Quebra de frases
(sentence boundary disambiguatio

## Etapa 7 — LLM local (Ollama + Llama 3.2)

Carregamos o **Llama 3.2** através do **Ollama**.
`temperature=0` torna as respostas determinísticas e menos propensas a inventar.

In [27]:
try:
    from langchain_ollama import ChatOllama
except ImportError:
    from langchain_community.chat_models import ChatOllama

llm = ChatOllama(
    base_url=OLLAMA_BASE_URL,
    model=OLLAMA_MODEL,
    temperature=0,
)

# Teste rápido
try:
    print("Resposta de teste do LLM:", llm.invoke("Responde apenas com a palavra: OK").content)
except Exception as e:
    print("Não foi possível contactar o Llama via Ollama. Revê a Etapa 1.")
    print("Detalhe técnico:", e)

Resposta de teste do LLM: OK


## Etapa 8 — Cadeia RAG (LCEL)

Montamos o *pipeline* em **LCEL** (LangChain Expression Language), tal como na aula **T11**:

```
{"context": retriever | formatar_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
```

O *prompt* instrui o modelo a responder **exclusivamente** com base no contexto e a
admitir quando não sabe — a principal defesa contra **alucinações** (**aula T09**).

In [28]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Prompt de sistema
SYSTEM_PROMPT = (
    "És um assistente académico especializado em Computação em Linguagem Natural. "
    "Responde à pergunta do utilizador usando EXCLUSIVAMENTE o contexto fornecido abaixo. "
    "Se a informação não constar do contexto, responde apenas: "
    "'Não encontrei essa informação no conteúdo indexado.'. "
    "Não inventes factos. Responde em português, de forma clara e fundamentada.\n\n"
    "Contexto recuperado:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("user", "{question}"),
])

# Junta o conteúdo dos chunks recuperados num único bloco de contexto
def formatar_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# Cadeia RAG completa (LCEL)
rag_chain = (
    {"context": retriever | formatar_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Cadeia RAG construída com sucesso.")

Cadeia RAG construída com sucesso.


## Etapa 9 — Perguntas e Respostas

A função `perguntar()` corre a cadeia RAG e mostra a resposta e as fontes
(chunks usados). Citando a proveniência de *Knowledge Citations* da **aula T09**.

In [30]:
def perguntar(pergunta, mostrar_fontes=True):
    resposta = rag_chain.invoke(pergunta)
    print("Pergunta:", pergunta, "\n")
    print("Resposta:")
    print(resposta)
    if mostrar_fontes:
        print("\n" + "-" * 80)
        print("Fontes (chunks recuperados):")
        for d in retriever.invoke(pergunta):
            print(f"  [chunk {d.metadata.get('chunk_id')}] {d.metadata.get('source')}")
    print("=" * 80)
    return resposta

_ = perguntar("Resume, em três pontos, o tema principal do conteúdo indexado.")
_ = perguntar("Quais os conceitos ou termos mais importantes mencionados?")

Pergunta: Resume, em três pontos, o tema principal do conteúdo indexado. 

Resposta:
O tema principal do conteúdo indexado é a Inteligência Artificial (IA), mais especificamente, áreas relacionadas à Processamento de Linguagem Natural (PLN). Os três pontos principais são:

1. **Processamento de Linguagem Natural**: A PLN é uma área da IA que se concentra em processar e entender linguagem natural humana, incluindo tarefas como a análise de subjetividade, reconhecimento de fala e quebra de frases.
2. **Análise de Subjetividade e Reconhecimento de Fala**: A análise de subjetividade (sentiment analysis) é uma técnica usada para extrair informações subjetivas de documentos, enquanto o reconhecimento de fala é um processo que determina a representação textual do discurso a partir de um clipe de som.
3. **Desafios e Técnicas da PLN**: A PLN é considerada uma das áreas mais difíceis da IA, conhecida como "AI-complete", devido à complexidade da linguagem natural e à falta de pausas entre palavr

## Etapa 10 — Demonstração anti-alucinação (RAG vs. LLM puro)

Comparamos a mesma pergunta:

- **LLM sozinho** → responde a partir da memória paramétrica (pode inventar);
- **Com RAG** → ancora-se no conteúdo indexado e, se a resposta lá não estiver, admite-o.

In [31]:
pergunta_fora_do_tema = "Qual é a capital da Austrália e qual a sua população exata hoje?"

print("=== LLM SOZINHO (memória paramétrica — pode alucinar) ===")
try:
    print(llm.invoke(pergunta_fora_do_tema).content)
except Exception as e:
    print("(LLM indisponível — revê a Etapa 1.)", e)

print("\n=== COM RAG (ancorado no conteúdo indexado) ===")
print(rag_chain.invoke(pergunta_fora_do_tema))

=== LLM SOZINHO (memória paramétrica — pode alucinar) ===
A capital da Austrália é Canberra. A população de Canberra é estimada em cerca de 415.000 habitantes, de acordo com o censo de 2021 realizado pela Australian Bureau of Statistics (ABS). No entanto, a área metropolitana de Canberra inclui também as áreas rurais e periurbanas, que têm uma população ainda maior.

É importante notar que a população de Canberra pode variar ao longo do tempo devido à migração e outros fatores demográficos. Além disso, a ABS publica novos resultados de censo regularmente, então a população exata de Canberra pode ter mudado desde o último censo em 2021.

Aqui estão algumas informações adicionais sobre a população de Canberra:

*   A cidade de Canberra tem uma área de aproximadamente 814 km².
*   A região metropolitana de Canberra inclui também as áreas rurais e periurbanas, que têm uma população estimada em cerca de 1 milhão de habitantes.
*   A população de Canberra é diversa e inclui pessoas de difere

## Etapa 11 — Operação Offline

**Recarrega-se a ChromaDB já persistida em disco** (sem scraping) e voltamos a responder.

In [33]:
from langchain_chroma import Chroma

def carregar_sistema_offline():
    # Recarrega a coleção a partir do disco
    vs = Chroma(
        collection_name=COLLECTION_NAME,
        persist_directory=CHROMA_DIR,
        embedding_function=embeddings,
    )
    ret = vs.as_retriever(search_kwargs={"k": TOP_K})
    chain = (
        {"context": ret | formatar_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain

rag_offline = carregar_sistema_offline()
print("Sistema recarregado do disco (modo offline). Resposta:\n")
print(rag_offline.invoke("Do que trata o documento indexado?"))

Sistema recarregado do disco (modo offline). Resposta:

O documento indexado parece tratar sobre a Computação em Linguagem Natural (NLP), um campo da ciência da computação que se concentra na análise e processamento de linguagem natural. O texto aborda vários tópicos relacionados à NLP, incluindo:

* Recuperação de informação (IR)
* Extração de informação (IE)
* Processamento de voz
* Identificação na língua materna
* Stemização
* Simplificação do texto
* Síntese de fala
* Revisão de texto
* Pesquisa em língua natural
* Expansão da consulta
* Pontuação de ensaio automatizado
* Truecasing
* Estatística

Além disso, o documento também menciona a Gramática estocástica e a padronização de recursos lexicais para facilitar a interoperabilidade entre programas PLN.

Em resumo, o documento indexado parece tratar sobre os fundamentos e aplicações da Computação em Linguagem Natural.


# Perguntas e Respostas

Faz perguntas em ciclo. Escreve "sair" para terminar.

In [17]:
# Sessão interativa simples (opcional).
while True:
    q = input("\nPergunta ('sair' para terminar): ").strip()
    if q.lower() in {"sair", "exit", "quit", ""}:
        print("Sessão terminada.")
        break
    perguntar(q, mostrar_fontes=False)

Pergunta: qual é o atual presidente de portugal? 

Resposta:
Não encontrei essa informação no conteúdo indexado.
Pergunta: responda  em uma frase curta o que é PNL 

Resposta:
PNL significa Processamento Natural de Linguagem, uma área da Computação que visa entender e gerenciar linguagem humana.
Sessão terminada.
